# Descarga nacional de resultados electorales Colombia 2026
**Autor: Nicolás Cardona - @cardonanl**

Versión optimizada para descargas nacionales con checkpoint por departamento.

**Elecciones disponibles:**
- `SE` → Senado
- `CA` → Cámara de Representantes
- `CN` → Consultas Interpartidistas
- `CT` → CITREP

> ⚠️ **Nota 2026:** Los códigos de departamento ya **no** son los códigos DANE de 2 dígitos.
> Ahora son códigos propios de 4 dígitos (ej: `3100` = Valle del Cauca).
> Ejecuta la celda 4b para ver todos los códigos disponibles.


---
## CELDA 1 — Configuración
### ⚠️ ÚNICO LUGAR DONDE HACER CAMBIOS


In [ ]:
# =============================================================================
# PARÁMETROS DEL USUARIO — MODIFICAR AQUÍ
# =============================================================================

# --- TIPO DE ELECCIÓN ---
# 'SE' → Senado | 'CA' → Cámara | 'CN' → Consultas | 'CT' → CITREP
TIPO_ELECCION = 'SE'   # <--- CAMBIAR AQUÍ

# --- SCOPE DE DESCARGA ---
# 'nacional'      → descarga todos los departamentos (con checkpoint)
# 'departamento'  → descarga un solo departamento
# 'municipio'     → descarga un solo municipio
SCOPE = 'nacional'   # <--- CAMBIAR AQUÍ

# --- CÓDIGO DEL TERRITORIO ---
# Solo aplica si SCOPE es 'departamento' o 'municipio'.
# Ejemplos departamentos (4 dígitos):
#   '0100'=Antioquia  '0300'=Atlántico  '1600'=Bogotá D.C.  '0500'=Bolívar
#   '0700'=Boyacá     '0900'=Caldas     '1100'=Cauca         '1200'=Cesar
#   '1300'=Córdoba    '1500'=Cundinamarca '1900'=Huila        '2100'=Magdalena
#   '2300'=Nariño     '2500'=N.Santander  '2600'=Quindío      '2400'=Risaralda
#   '2700'=Santander  '2800'=Sucre        '2900'=Tolima       '3100'=Valle
#   '4000'=Arauca     '4400'=Caquetá      '4600'=Casanare     '4800'=La Guajira
#   '5000'=Guainía    '5200'=Meta         '5400'=Guaviare     '5600'=San Andrés
#   '6000'=Amazonas   '6400'=Putumayo     '6800'=Vaupés       '7200'=Vichada
#   '8800'=Consulados
CODIGO_TERRITORIO = '3100'   # <--- CAMBIAR AQUÍ (ignorado si SCOPE='nacional')

# --- FILTRO OPCIONAL POR CANDIDATO ---
CODIGO_CANDIDATO = None   # <--- codcan específico, o None para todos

# --- NOMENCLATOR ---
USAR_NOMENCLATOR_LOCAL = True
RUTA_NOMENCLATOR_LOCAL = 'nomenclator_2026.json'

# --- DESCARGA ---
DELAY_SEG     = 1.5   # segundos entre requests (no bajar de 1.0)
VERBOSE_CADA  = 10    # imprimir progreso cada N municipios
CARPETA_SALIDA = f'resultados_{TIPO_ELECCION}_2026'  # carpeta para CSVs parciales

# =============================================================================
print('Configuración cargada:')
print(f'  Tipo de elección : {TIPO_ELECCION}')
print(f'  Scope            : {SCOPE}')
if SCOPE != 'nacional':
    print(f'  Código territorio: {CODIGO_TERRITORIO}')
print(f'  Filtro candidato : {CODIGO_CANDIDATO or "Todos"}')
print(f'  Nomenclator local: {USAR_NOMENCLATOR_LOCAL}')
print(f'  Carpeta salida   : {CARPETA_SALIDA}/')


---
## CELDA 2 — Imports y constantes


In [ ]:
import requests
import pandas as pd
import json
import time
import glob
from pathlib import Path
from collections import Counter

BASE_URL               = 'https://resultados.registraduria.gov.co'
NOMENCLATOR_URL        = f'{BASE_URL}/json/nomenclator.json'
RESULTADOS_URL_TEMPLATE = BASE_URL + '/json/ACT/{tipo}/{codigo}.json'

NIVEL_NOMBRES = {
    1: 'País',
    2: 'Departamento',
    3: 'Municipio',
    4: 'Zona',
    5: 'Corregimiento',
    6: 'Puesto de votación',
}

SIGLA_A_ELEC = {'SE': 1, 'CA': 2, 'CN': 6, 'CT': 7}

COL_RENAME = {
    'i': 'id', 'n': 'nombre', 'co': 'codigo', 's': 'slug',
    'l': 'nivel', 'p': 'padre', 'r': 'refs', 'h': 'hijos'
}

HEADERS_BROWSER = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Referer'   : 'https://resultados.registraduria.gov.co/',
    'Accept'    : 'application/json, text/plain, */*',
}

print('Librerías importadas correctamente.')


---
## CELDA 3 — Cargar nomenclator


In [ ]:
def cargar_nomenclator(usar_local=False, ruta_local='nomenclator_2026.json'):
    """Carga el nomenclator y devuelve (data, ambitos) para el TIPO_ELECCION configurado."""
    if usar_local:
        ruta = Path(ruta_local)
        if not ruta.exists():
            raise FileNotFoundError(
                f"No se encontró '{ruta_local}'. "
                "Descárgalo desde: https://resultados.registraduria.gov.co/json/nomenclator.json"
            )
        print(f'Cargando nomenclator desde archivo local: {ruta_local}')
        with open(ruta, 'r', encoding='utf-8') as f:
            data = json.load(f)
    else:
        print(f'Descargando nomenclator desde {NOMENCLATOR_URL} ...')
        response = requests.get(NOMENCLATOR_URL, headers=HEADERS_BROWSER, timeout=60)
        response.raise_for_status()
        data = response.json()
        with open('nomenclator_2026.json', 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False)
        print("Guardado localmente como 'nomenclator_2026.json'")

    elec_num = SIGLA_A_ELEC.get(TIPO_ELECCION, 1)
    ambitos  = {}
    for grupo in data.get('amb', []):
        if grupo.get('elec') == elec_num:
            for a in grupo.get('ambitos', []):
                ambitos[str(a['i'])] = a
            break

    print(f"Nomenclator cargado — {len(ambitos):,} ámbitos para '{TIPO_ELECCION}'")
    return data, ambitos


data, ambitos = cargar_nomenclator(
    usar_local=USAR_NOMENCLATOR_LOCAL,
    ruta_local=RUTA_NOMENCLATOR_LOCAL
)


---
## CELDA 4 — Explorar estructura del nomenclator


In [ ]:
def explorar_nomenclator(ambitos):
    conteo = Counter(a.get('l') for a in ambitos.values())
    print(f'Total ámbitos: {len(ambitos):,}\n')
    for nivel in sorted(conteo):
        nombre  = NIVEL_NOMBRES.get(nivel, f'Nivel {nivel}')
        ejemplo = next((a for a in ambitos.values() if a.get('l') == nivel), {})
        print(f"  Nivel {nivel} ({nombre}): {conteo[nivel]:>6,} | ejemplo: co='{ejemplo.get('co','')}' — {ejemplo.get('n','')}")

explorar_nomenclator(ambitos)


---
## CELDA 4b — Consultar códigos disponibles
> ⚠️ Códigos de departamento en 2026 son de 4 dígitos (`'3100'` = Valle). No son los códigos DANE de 2023.


In [ ]:
def listar_territorios(ambitos, nivel, filtro_prefijo=None):
    filas  = []
    vistos = set()
    for item in ambitos.values():
        if item.get('l') != nivel:
            continue
        codigo = str(item.get('co', ''))
        if filtro_prefijo and not codigo.startswith(filtro_prefijo):
            continue
        if codigo in vistos:
            continue
        vistos.add(codigo)
        filas.append({'codigo': codigo, 'nombre': item.get('n', ''), 'nivel': nivel})
    return pd.DataFrame(filas).sort_values('codigo').reset_index(drop=True)


print('DEPARTAMENTOS DISPONIBLES (nivel 2)')
print('=' * 50)
df_deptos = listar_territorios(ambitos, nivel=2)
print(df_deptos.to_string(index=False))


In [ ]:
# Ver municipios de un departamento específico
codigo_depto = '3100'   # <--- CAMBIAR AQUÍ

df_municipios = listar_territorios(ambitos, nivel=3, filtro_prefijo=codigo_depto)
print(f"MUNICIPIOS DE '{codigo_depto}' ({len(df_municipios)} municipios)")
print('=' * 60)
print(df_municipios.to_string(index=False))


---
## CELDA 5 — Funciones de extracción y descarga


In [ ]:
def extraer_municipios(ambitos, codigo_territorio=None):
    """
    Extrae municipios (nivel 3) del nomenclator.
    Si codigo_territorio es None, retorna TODOS los municipios (descarga nacional).
    """
    municipios = {}
    for item in ambitos.values():
        if item.get('l') != 3:
            continue
        codigo = str(item.get('co', ''))
        if codigo_territorio and not codigo.startswith(codigo_territorio):
            continue
        municipios[codigo] = item
    return list(municipios.values())


def descargar_votos(puestos, tipo_eleccion, codigo_candidato=None,
                    delay_seg=1.5, verbose_cada=10):
    """
    Descarga votos para una lista de municipios.
    Retorna (DataFrame, lista_errores).
    """
    codigos        = [str(p.get('co', '')) for p in puestos]
    nombres_puesto = {str(p.get('co', '')): p.get('n', 'Desconocido') for p in puestos}
    total          = len(codigos)
    filas          = []
    errores        = []

    for idx, codigo in enumerate(codigos):
        if idx % verbose_cada == 0 or idx == total - 1:
            pct = (idx + 1) / total * 100
            print(f'  [{idx+1}/{total}] ({pct:.0f}%) {codigo}')

        url = RESULTADOS_URL_TEMPLATE.format(tipo=tipo_eleccion, codigo=codigo)
        try:
            resp = requests.get(url, headers=HEADERS_BROWSER, timeout=30)
            if resp.status_code != 200:
                errores.append({'codigo': codigo, 'status': resp.status_code})
                continue

            data_json = resp.json()
            for camara in data_json.get('camaras', []):
                for partido in camara.get('partotabla', []):
                    acto   = partido.get('act', {})
                    codpar = acto.get('codpar', '')
                    for candidato in acto.get('cantotabla', []):
                        if codigo_candidato and str(candidato.get('codcan', '')) != str(codigo_candidato):
                            continue
                        row = dict(candidato)
                        row['codpar']        = codpar
                        row['codigo_puesto'] = codigo
                        row['nombre_puesto'] = nombres_puesto.get(codigo, 'Desconocido')
                        filas.append(row)

            time.sleep(delay_seg)

        except requests.exceptions.RequestException as e:
            errores.append({'codigo': codigo, 'error': str(e)})

    df = pd.DataFrame(filas)
    if errores:
        print(f'  ⚠️  {len(errores)} errores')
    print(f'  ✅ {len(df):,} filas | {total} municipios procesados')
    return df, errores


def enriquecer_df(df, ambitos):
    """Añade nombre, slug y jerarquía territorial al DataFrame de votos."""
    df = df.copy()
    mapeo_nombre = {str(a.get('co','')): a.get('n','') for a in ambitos.values() if a.get('l') == 3}
    mapeo_slug   = {str(a.get('co','')): a.get('s','') for a in ambitos.values() if a.get('l') == 3}
    mapeo_depto  = {str(a.get('co','')): a.get('n','') for a in ambitos.values() if a.get('l') == 2}

    df['nombre_municipio'] = df['codigo_puesto'].map(mapeo_nombre).fillna('Desconocido')
    df['slug_municipio']   = df['codigo_puesto'].map(mapeo_slug).fillna('Desconocido')
    df['cod_depto']        = df['codigo_puesto'].astype(str).str[:4]
    df['cod_municipio']    = df['codigo_puesto'].astype(str).str[:7]
    df['nombre_depto']     = df['cod_depto'].map(mapeo_depto).fillna('Desconocido')
    return df


print('Funciones cargadas.')


---
## CELDA 6 — Descarga
Ejecuta según el `SCOPE` configurado en la celda 1:
- **`nacional`** → descarga todos los departamentos con checkpoint (puede tardar 1-3 horas)
- **`departamento`** → descarga un solo departamento
- **`municipio`** → descarga un solo municipio

> 💡 Si Colab se desconecta durante una descarga nacional, simplemente vuelve a ejecutar esta celda — los departamentos ya descargados se saltan automáticamente.


In [ ]:
Path(CARPETA_SALIDA).mkdir(exist_ok=True)

lista_errores_global = []

# =============================================================================
if SCOPE == 'nacional':
# =============================================================================
    df_deptos = listar_territorios(ambitos, nivel=2)
    print(f'Descarga nacional — {len(df_deptos)} departamentos, {len(extraer_municipios(ambitos)):,} municipios totales')
    print(f'Carpeta de salida: {CARPETA_SALIDA}/\n')

    for i, row in df_deptos.iterrows():
        cod_depto    = row['codigo']
        nombre_depto = row['nombre']
        archivo      = f"{CARPETA_SALIDA}/{TIPO_ELECCION}_{cod_depto}.csv"

        if Path(archivo).exists():
            print(f'⏭️  [{i+1}/{len(df_deptos)}] {nombre_depto} — ya descargado')
            continue

        print(f'\n📥 [{i+1}/{len(df_deptos)}] {nombre_depto} ({cod_depto})')
        municipios = extraer_municipios(ambitos, cod_depto)
        print(f'   {len(municipios)} municipios')

        df, errores = descargar_votos(
            puestos          = municipios,
            tipo_eleccion    = TIPO_ELECCION,
            codigo_candidato = CODIGO_CANDIDATO,
            delay_seg        = DELAY_SEG,
            verbose_cada     = VERBOSE_CADA,
        )
        lista_errores_global.extend(errores)

        df = enriquecer_df(df, ambitos)
        df.to_csv(archivo, index=False, encoding='utf-8-sig')
        print(f'   💾 Guardado: {archivo}')

    print('\n🏁 Descarga nacional completa.')
    if lista_errores_global:
        print(f'⚠️  Total errores acumulados: {len(lista_errores_global)}')
        pd.DataFrame(lista_errores_global).to_csv(f'{CARPETA_SALIDA}/errores.csv', index=False)

# =============================================================================
elif SCOPE == 'departamento':
# =============================================================================
    municipios = extraer_municipios(ambitos, CODIGO_TERRITORIO)
    print(f'Descargando departamento {CODIGO_TERRITORIO} — {len(municipios)} municipios')

    df_votos, lista_errores_global = descargar_votos(
        puestos          = municipios,
        tipo_eleccion    = TIPO_ELECCION,
        codigo_candidato = CODIGO_CANDIDATO,
        delay_seg        = DELAY_SEG,
        verbose_cada     = VERBOSE_CADA,
    )
    df_votos = enriquecer_df(df_votos, ambitos)

# =============================================================================
elif SCOPE == 'municipio':
# =============================================================================
    municipios = extraer_municipios(ambitos, CODIGO_TERRITORIO)
    if not municipios:
        raise ValueError(f"No se encontró municipio con código '{CODIGO_TERRITORIO}'")

    print(f"Descargando municipio {CODIGO_TERRITORIO} — {municipios[0].get('n', '')}")
    df_votos, lista_errores_global = descargar_votos(
        puestos          = municipios,
        tipo_eleccion    = TIPO_ELECCION,
        codigo_candidato = CODIGO_CANDIDATO,
        delay_seg        = DELAY_SEG,
        verbose_cada     = 1,
    )
    df_votos = enriquecer_df(df_votos, ambitos)

else:
    raise ValueError(f"SCOPE debe ser 'nacional', 'departamento' o 'municipio'. Valor recibido: '{SCOPE}'")


---
## CELDA 7 — Consolidar CSVs parciales en un archivo nacional
Solo necesaria si `SCOPE = 'nacional'`. Une todos los archivos parciales en uno solo.


In [ ]:
archivos = sorted(glob.glob(f'{CARPETA_SALIDA}/{TIPO_ELECCION}_*.csv'))
print(f'Archivos encontrados: {len(archivos)}')
for a in archivos:
    print(f'  {a}')

df_nacional = pd.concat(
    [pd.read_csv(f) for f in archivos],
    ignore_index=True
)

nombre_final = f'{TIPO_ELECCION}_2026_NACIONAL.csv'
df_nacional.to_csv(nombre_final, index=False, encoding='utf-8-sig')
print(f'\n✅ Exportado: {nombre_final}')
print(f'   {df_nacional.shape[0]:,} filas × {df_nacional.shape[1]} columnas')
df_nacional.head()


---
## CELDA 8 — (Opcional) Agregar nombres de partidos


In [ ]:
MAPEO_PARTIDOS = {
    '2':    'Partido Liberal',
    '3':    'Partido Conservador',
    '4':    'Cambio Radical',
    '5':    'Partido Verde',
    '9':    'Partido de la U',
    '12':   'Centro Democrático',
    '16':   'Colombia Renaciente',
    '18':   'Dignidad y Compromiso',
    '20':   'Nuevo Liberalismo',
    '21':   'Salvación Nacional',
    '22':   'Partido Verde Oxígeno',
    '24':   'Liga Gobernantes Anticorrupción',
    '26':   'Partido Ecologista Colombiano',
    '27':   'Fuerza de la Paz',
    '29':   'Nueva Fuerza Democrática',
    '31':   'Independientes',
    '33':   'Creemos',
    '35':   'Gente en Movimiento',
    '36':   'Fuerza Ciudadana',
    '37':   'Colombia Humana',
    '2484': 'Pacto Histórico Colombia Puede',
    '2677': 'Nuevo Liberalismo - Nueva Fuerza Democrática',
    '2717': 'Pacto Histórico',
    '2803': 'Pacto Histórico',
    '2997': 'Acuerdo de Coalición',
    '1773': 'Fuerza de la Paz - Mais',
}

# Aplica sobre df_votos (scope depto/municipio) o df_nacional (scope nacional)
df_target = df_nacional if 'df_nacional' in dir() else df_votos
if 'codpar' in df_target.columns:
    df_target['partido'] = df_target['codpar'].astype(str).map(MAPEO_PARTIDOS).fillna('Desconocido')
    print('Partidos únicos encontrados:')
    print(df_target[['codpar', 'partido']].drop_duplicates().sort_values('codpar').to_string(index=False))


---
## CELDA 9 — Utilidades de diagnóstico


In [ ]:
def inspeccionar_json_municipio(codigo_municipio, tipo_eleccion):
    """Muestra las claves del JSON de resultados para diagnosticar cambios de estructura."""
    url  = RESULTADOS_URL_TEMPLATE.format(tipo=tipo_eleccion, codigo=codigo_municipio)
    resp = requests.get(url, headers=HEADERS_BROWSER, timeout=30)
    resp.raise_for_status()
    d = resp.json()
    print('Claves top-level:', list(d.keys()))
    for camara in d.get('camaras', [])[:1]:
        print('Claves camara:', list(camara.keys()))
        for partido in camara.get('partotabla', [])[:1]:
            print('Claves partido:', list(partido.keys()))
            acto = partido.get('act', {})
            print('Claves act:', list(acto.keys()))
            for candidato in acto.get('cantotabla', [])[:1]:
                print('Claves candidato:', list(candidato.keys()))


def resumen_errores():
    """Muestra un resumen de los errores ocurridos durante la descarga."""
    if not lista_errores_global:
        print('Sin errores.')
        return
    df_err = pd.DataFrame(lista_errores_global)
    print(f'Total errores: {len(df_err)}')
    if 'status' in df_err.columns:
        print(df_err['status'].value_counts().to_string())
    print(df_err.head(20).to_string(index=False))


# Ejemplos de uso (descomentar):
# inspeccionar_json_municipio('3100001', TIPO_ELECCION)
# resumen_errores()
